# Document Intelligence Platform - Colab eval runner

Runtime: **GPU (T4)**. This clones the repo, brings up Ollama (Qwen2.5-3B on GPU) and a
Qdrant server, runs the full pipeline over the 120-doc eval set, and produces `artifacts/`.

Set `REPO_URL` below (push the project to GitHub first), then Runtime -> Run all.

In [ ]:
REPO_URL = "https://github.com/PrayuktaKute/document_enterprise_platform.git"
import os
if not os.path.isdir('/content/document_enterprise_platform'):
    !git clone $REPO_URL /content/document_enterprise_platform
%cd /content/document_enterprise_platform

In [ ]:
!bash scripts/colab_bootstrap.sh

### Data
If `data/raw` is not in the repo, regenerate it (SROIE + CUAD download, synthetic PDFs).

In [ ]:
import os
# manifest.jsonl is committed but the raw documents are not -- regenerate them.
if not os.path.isdir('data/raw/invoices') or not os.listdir('data/raw/invoices'):
    !python scripts/fetch_data.py --invoices 30 --contracts 30
    !python scripts/gen_synthetic.py --purchase-orders 30 --medical 30 --seed 7
    !python scripts/prepare_eval.py

In [ ]:
!WORKERS=4 bash scripts/run_all_eval.sh

In [ ]:
import json
print(open('artifacts/eval_report.md').read())
print(json.dumps(json.load(open('artifacts/retrieval_metrics.json')), indent=2)[:1500])

In [ ]:
from google.colab import files
files.download('artifacts_bundle.zip')